In [ ]:
# ---- 1. Locate code + data -------------------------------------------------
import os, sys, shutil, glob
from pathlib import Path

def _first(cands):
    for c in cands:
        for hit in glob.glob(c):
            p = Path(hit)
            if p.exists(): return p
    return None

DATA = _first(['/kaggle/input/birdclef-2026',
               '/kaggle/input/competitions/birdclef-2026',
               '/kaggle/input/*birdclef*2026*'])
assert DATA is not None and (DATA/'taxonomy.csv').exists(), 'birdclef-2026 not attached'

CODE = _first(['/kaggle/input/birdclef2026-code',
               '/kaggle/input/datasets/ahmedsherif382/birdclef2026-code',
               '/kaggle/input/datasets/*/birdclef2026-code',
               '/kaggle/input/*/birdclef2026-code',
               '/kaggle/input/*birdclef2026-code*'])
assert CODE is not None, 'birdclef2026-code not attached'

def _resolve(parent, name, marker):
    for cand in (parent/name/name, parent/name):
        if (cand/marker).exists(): return cand
    raise FileNotFoundError(f'{name}/{marker} under {parent}')

SRC = _resolve(CODE, 'src', 'taxonomy.py')
DP  = _resolve(CODE, 'data_prep', 'make_folds.py')
os.environ['BIRDCLEF_DATA'] = str(DATA)

WORK = Path('/kaggle/working')
for s, n in ((SRC,'src'), (DP,'data_prep')):
    d = WORK/n
    if d.exists(): shutil.rmtree(d)
    shutil.copytree(s, d)
for m in list(sys.modules):
    if m.split('.')[0] in ('src','data_prep'): del sys.modules[m]
sys.path[:] = [str(WORK)] + [p for p in sys.path if p != str(WORK)]

# locate best.pt
CKPT_HITS = sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True))
assert CKPT_HITS, 'no best.pt under /kaggle/input - attach the model dataset'
CKPT = CKPT_HITS[0]
print('data :', DATA)
print('code :', CODE)
print('ckpt :', CKPT)

In [ ]:
# ---- 2. Load model ---------------------------------------------------------
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader

from src.models import build_model
from src.taxonomy import num_classes, class_to_idx, class_list
from src.metrics import birdclef_roc_auc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ck = torch.load(CKPT, map_location=device, weights_only=False)
cfg = ck.get('cfg', {})
print(f'ckpt: auc={ck.get("auc",float("nan")):.4f}  epoch={ck.get("epoch")}  '
      f'backbone={cfg.get("backbone","tf_efficientnet_b0.ns_jft_in1k")}')

model = build_model('timm',
                    backbone=cfg.get('backbone','tf_efficientnet_b0.ns_jft_in1k'),
                    num_classes=num_classes(),
                    pretrained=False,
                    drop_rate=cfg.get('drop_rate',0.3)).to(device)
missing, unexpected = model.load_state_dict(ck['model'], strict=False)
if missing:    print('  missing keys:', len(missing), missing[:3])
if unexpected: print('  unexpected:', len(unexpected), unexpected[:3])
model.eval(); torch.set_grad_enabled(False)
print('model on', device)

In [ ]:
# ---- 3. Build evaluation index --------------------------------------------
# Eval set 1: ALL labeled soundscape segments (real distribution)
# Eval set 2: random sample of focal clips, top-energy crop (sanity baseline)
from src.ogg_dataset import OggOnTheFlyDataset, build_index
from data_prep.make_folds import build_folds

FOLDS = WORK/'folds.csv'
fdf = build_folds(n_folds=5, seed=42)
fdf.to_csv(FOLDS, index=False)
print('rebalanced fold sizes (segments):')
labels = pd.read_csv(DATA/'train_soundscapes_labels.csv')
seg_per = labels.groupby('filename').size()
fold_seg = fdf.assign(n_seg=fdf['filename'].map(seg_per)).groupby('fold')['n_seg'].sum()
print(fold_seg.to_string())

all_idx = build_index(folds_csv=FOLDS)
ss_full   = all_idx[all_idx['source']=='soundscape'].reset_index(drop=True)
focal_smp = all_idx[all_idx['source']=='focal'].sample(n=min(2000, (all_idx['source']=='focal').sum()),
                                                       random_state=0).reset_index(drop=True)
print(f'\nSoundscape eval: {len(ss_full)} segments  ({ss_full["fold"].value_counts().to_dict()})')
print(f'Focal eval     : {len(focal_smp)} clips')

In [ ]:
# ---- 4. Evaluate -----------------------------------------------------------
from tqdm.auto import tqdm

def evaluate_split(df, label):
    ds = OggOnTheFlyDataset(df, train=False)
    dl = DataLoader(ds, batch_size=128, shuffle=False, num_workers=4,
                    pin_memory=True, persistent_workers=False)
    preds, tgts = [], []
    for b in tqdm(dl, desc=label, leave=False):
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            logits = model(b['mel'].to(device, non_blocking=True))
        preds.append(torch.sigmoid(logits).float().cpu().numpy())
        tgts.append(b['target'].numpy())
    P = np.concatenate(preds); T = np.concatenate(tgts)
    Tb = (T > 0.5).astype(np.float32)
    auc = birdclef_roc_auc(Tb, P)
    n_classes_present = int((Tb.sum(axis=0) > 0).sum())
    print(f'{label:<22} N={len(P):<5} classes_present={n_classes_present:<3} macro_AUC={auc:.4f}')
    return auc, P, T

ss_auc, P_ss, T_ss = evaluate_split(ss_full, 'soundscape (FULL)')
focal_auc, _, _    = evaluate_split(focal_smp, 'focal sample')

# Per-fold breakdown on soundscapes (so you can see if val_fold=0 was unlucky):
print('\nper-fold soundscape AUC:')
for f in sorted(ss_full['fold'].unique()):
    mask = (ss_full['fold']==f).values
    if mask.sum() < 20: continue
    Tb = (T_ss[mask] > 0.5).astype(np.float32)
    n_present = int((Tb.sum(axis=0) > 0).sum())
    print(f'  fold {f}: N={mask.sum():<4} classes={n_present:<3} AUC={birdclef_roc_auc(Tb, P_ss[mask]):.4f}')

In [ ]:
# ---- 5. Summary ------------------------------------------------------------
import json
summary = {
    'ckpt': str(CKPT),
    'reported_train_auc': float(ck.get('auc', float('nan'))),
    'true_soundscape_auc_full': float(ss_auc),
    'focal_auc_2k_sample': float(focal_auc),
    'n_soundscape_segs': int(len(ss_full)),
    'n_focal_eval': int(len(focal_smp)),
}
print(json.dumps(summary, indent=2))
(WORK/'eval_summary.json').write_text(json.dumps(summary, indent=2))
print('\nsaved /kaggle/working/eval_summary.json')
print('\nInterpretation:')
print('  - true_soundscape_auc_full is your REAL signal (LB tracks soundscape distribution)')
print('  - if it is much higher than reported_train_auc -> the training val fold was unlucky')
print('  - focal AUC will be higher than soundscape (cleaner audio); ignore for LB calibration')